# 台灣好聲音（PitchLab）雲端 AI 分軌與歌詞生成器
### 操作只要 3 步（免連雲端硬碟，直接下載到手機）：
1. 上方選單 **「執行階段」→「變更執行階段類型」→ 選 `T4 GPU`**（分軌＋歌詞只要約 30 秒）。
2. 點擊下方 **「步驟 1」左側的 ▶ 播放鈕**（安裝 AI 分軌與 Whisper 歌詞工具，約 1 分半）。
3. 點擊 **「步驟 2」左側的 ▶ 播放鈕**：可選擇 `LYRICS_MODE` 自動生成歌詞、貼上純文字歌詞對齊時間軸、或上傳自己的 `.lrc` 檔，跑完會**自動下載 `.zip` 歌包**！

In [ ]:
#@title 步驟 1：安裝 AI 分軌與歌詞環境（點左側 ▶ 執行一次即可）
!git clone https://github.com/bard0228/pitchlab.git /content/pitchlab 2>/dev/null || (cd /content/pitchlab && git pull)
!pip -q install -r /content/pitchlab/pipeline/requirements.txt
!apt-get -qq install -y ffmpeg
print('✅ 環境安裝完成，請接著執行步驟 2！')

In [ ]:
#@title 步驟 2：上傳歌曲或貼 YouTube 網址 → 自動產出並下載 .zip 歌包
YOUTUBE_URL = "" #@param {type:"string"}
SONG_TITLE = "" #@param {type:"string"}
LYRICS_MODE = "whisper" #@param ["whisper", "align_text", "upload_lrc", "none"]
PLAIN_LYRICS = "" #@param {type:"string"}

from google.colab import files
from pathlib import Path
import subprocess

out_dir = Path('/content/pitchlab_out')
out_dir.mkdir(exist_ok=True)

cmd = ['python3', '/content/pitchlab/pipeline/process_song.py', '--separator', 'demucs']
if YOUTUBE_URL.strip():
    cmd += ['--youtube', YOUTUBE_URL.strip()]
else:
    print('👇 請點下方「選擇檔案」按鈕上傳 MP3 / WAV / M4A：')
    uploaded = files.upload()
    src_file = next(iter(uploaded.keys()))
    cmd += ['--input', src_file]
    if not SONG_TITLE.strip():
        SONG_TITLE = Path(src_file).stem

if LYRICS_MODE == 'upload_lrc':
    print('👇 請上傳對應的 .lrc 動態歌詞檔：')
    lrc_up = files.upload()
    lrc_file = next(iter(lrc_up.keys()))
    cmd += ['--lrc', lrc_file]
elif LYRICS_MODE == 'align_text':
    txt_path = out_dir / 'plain_lyrics.txt'
    if PLAIN_LYRICS.strip():
        txt_path.write_text(PLAIN_LYRICS.replace('\\n', '\n'), encoding='utf-8')
    else:
        print('👇 請上傳純文字歌詞 .txt 檔：')
        txt_up = files.upload()
        txt_path = Path(next(iter(txt_up.keys())))
    cmd += ['--lyrics-mode', 'align', '--lyrics-text', str(txt_path)]
elif LYRICS_MODE == 'whisper':
    cmd += ['--lyrics-mode', 'whisper']

if SONG_TITLE.strip():
    cmd += ['--title', SONG_TITLE.strip()]

out_name = (SONG_TITLE.strip() or 'song_package') + '.zip'
out_zip = out_dir / out_name
cmd += ['--out', str(out_zip)]

print('🎵 AI 分軌與歌詞處理中，請稍候...')
res = subprocess.run(cmd, check=False)
if res.returncode == 0 and out_zip.exists():
    print('✅ 完成！正在下載歌包：', out_zip.name)
    files.download(str(out_zip))
else:
    print('❌ 處理失敗，請查看上方錯誤訊息（若 YouTube 被擋請留空 YOUTUBE_URL 改用上傳本機音檔）。')